## initialize

In [ ]:
import os,sys,subprocess,glob,importlib,pickle,itertools
from datetime import datetime
import xarray as xr
import numpy as np
import pandas as pd
import scipy

sys.path.append('create_rea_ensembles')
from ensembles.ensemble_GKLT import ensemble_GKLT,get_weight_for_selection
from experiment_configuration.experiment import experiment

def import_from(module, name):
    module = __import__(module, fromlist=[name])
    return getattr(module, name)

%load_ext autoreload
%autoreload 2

In [ ]:
data_dir = 'PATH_TO_SPECIFY' # need to specify this

In [3]:
climates = ['ssp370-2025', 'piControl']

## prepare data

In [4]:
ensembles = {}

In [5]:
ensembles['piControl-initial'] = dict(
    label = 'piControl initial',
    data = {},
)
ensembles['ssp370-2025-initial'] = dict(
    label = 'ssp370-2025 initial',
    data = {},
)
for ens_name, ens in ensembles.items():
    ensembles[ens_name]['data_initial'] = ens['data']


In [6]:
for experiment_identifier in [f"c{i}" for i in [1,2,3,4,5,10]] + [f"p{i}" for i in range(1,7)]:
    exp = experiment(import_from(f'experiment_configuration.{experiment_identifier}', 'config'))
    ensembles[f"{exp.initial_conditions_name}-x{exp.experiment_new_identifier}"] = dict(
        label = f"{exp.initial_conditions_name}-x{exp.experiment_new_identifier} k: {exp.k}",
        exp = exp,
        data = {},
        data_initial = ensembles[f"{exp.initial_conditions_name}-initial"]['data']
    )

In [ ]:
for experiment_identifier in ['c7_dry','c8_dry','c12_dry', 'c6_dry','c9_dry', 'c11_dry']:
    exp = experiment(import_from(f'experiment_configuration.{experiment_identifier}', 'config'))
    ensembles[f"{exp.initial_conditions_name}-x{exp.experiment_new_identifier}"] = dict(
        label = f"{exp.initial_conditions_name}-x{exp.experiment_new_identifier} k: {exp.k}",
        exp = exp,
        data = {},
        data_initial = ensembles[f"{exp.initial_conditions_name.split('-wet')[0].split('-dry')[0]}-initial"]['data']
    )

In [ ]:
### tas
for ens_name in ensembles.keys():
    print(ens_name)
    ensembles[ens_name]['data']['tas'] = xr.open_mfdataset(
        f"{data_dir}/{ens_name}/day/atmos/tas-reg/*/*", 
        concat_dim='sim', combine='nested')['tas'].load()

In [ ]:
for ens_name in ensembles.keys():
    if 'initial' not in ens_name:
        ensembles[ens_name]['data']['weight'] = xr.open_mfdataset(f"{data_dir}/{ens_name}/meta/weight_seas*")['weight'].load()        
        ensembles[ens_name]['data']['prob'] = xr.open_mfdataset(f"{data_dir}/{ens_name}/meta/probability_seas*")['probability'].load()        

In [ ]:
for ens_name in ensembles.keys():
    if 'initial' in ens_name:
        ensembles[ens_name]['data']['weight'] = ensembles[ens_name]['data']['tas'][:,0].squeeze() * 0 + 1.

        ensembles[ens_name]['data']['prob'] = ensembles[ens_name]['data']['tas'][:,0].squeeze() * 0
        abso = ensembles[ens_name]['data']['tas'].mean('time')
        ensembles[ens_name]['data']['prob'][:] = np.array([(abso >= a).astype(float).mean() for a in abso])

## add data

### vars 1D

In [ ]:
for var,realm in dict(
    hfls = 'atmos',
    hfss = 'atmos',
    pr = 'atmos',
    mrsos = 'land',
).items():
    print(var)
    for ens_name in ensembles.keys():
        try:
            ensembles[ens_name]['data'][var] = xr.open_mfdataset(
                f"{data_dir}/{ens_name}/day/{realm}/{var}-reg/*/*", 
                concat_dim='sim', combine='nested')[var].load()
            print(ens_name)
        except:
            print(f'{ens_name} -> missing')

## save pickle

In [17]:
with open('REA_heat_1D.pkl', 'wb') as fl:
    pickle.dump({ens_name : {k:v for k,v in ens.items() if k != 'exp'} for ens_name,ens in ensembles.items()}, fl)